# Pré-processamento de Dados — TOL19

Dataset original: 1752256927-TOL19-tests.xlsx

## Objetivo
Realizar o diagnóstico, a limpeza e a preparação dos dados do conjunto TOL19.

Os nomes de colunas e os valores categóricos são traduzidos para o inglês logo após a reestruturação, de modo que todas as etapas seguintes (diagnóstico e análise) trabalham com o esquema em inglês.

fonte metodológica: https://wesmckinney.com/book/data-cleaning 


## 1 Reestruturação do Dataset TOL-BR

A estrutura do raw dataset é:

- Colunas 0–7: Dados demográficos (ID, Sexo, Data nascimento, Idade, Escolaridade, Estado, Data aplicação)
- Colunas 8 em diante: Blocos repetidos de 9 colunas por tentativa (Rodada, Tentativa, Pontuação, Tempo de início, Tempo de fim, Número de movimentos, Descrição dos movimentos, Tempo do primeiro clique correto, Tempo do primeiro clique errado)

A seção abaixo transforma o formato wide (colunas repetidas por tentativa) em formato long (uma linha por tentativa).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_raw = pd.read_excel('../data/external/1752256927-TOL19-tests.xlsx')

df_raw.head()

In [ ]:
# Colunas demográficas (fixas por participante)
demo_cols = df_raw.columns[:7].tolist()

# Cada bloco de tentativa tem 9 colunas
tentative_columns = [
    'Rodada',
    'Tentativa',
    'Pontuação',
    'Tempo de início',
    'Tempo de fim',
    'Número de movimentos',
    'Descrição dos movimentos',
    'Tempo do primeiro clique correto',
    'Tempo do primeiro clique errado'
]
block_size = len(tentative_columns)

print('Colunas fixas:', len(demo_cols))
print('Colunas por tentativa:', block_size)
print('Número de blocos:', (df_raw.shape[1] - len(demo_cols)) // block_size)

In [ ]:
# O arquivo pode já estar no formato long (16 colunas: 7 fixas + 9 por tentativa).
if df_raw.shape[1] == len(demo_cols) + block_size:
    df = df_raw.copy()
    df.columns = demo_cols + tentative_columns
else:
    # Formato wide: reorganizar os blocos de tentativa em linhas.
    n_blocks = (df_raw.shape[1] - len(demo_cols)) // block_size

    demo_values = df_raw.iloc[:, :len(demo_cols)].values
    trial_values = df_raw.iloc[:, len(demo_cols):len(demo_cols) + n_blocks * block_size].values

    trial_reshaped = trial_values.reshape(len(df_raw), n_blocks, block_size)
    demo_repeated = np.repeat(demo_values, n_blocks, axis=0)
    trials_flat = trial_reshaped.reshape(-1, block_size)

    df = pd.DataFrame(
        np.column_stack([demo_repeated, trials_flat]),
        columns=demo_cols + tentative_columns
    )

# Remover linhas onde Rodada é nula (participantes com menos tentativas)
df = df.dropna(subset=['Rodada']).reset_index(drop=True)

print(f'Shape (formato long): {df.shape}')
df.head(10)

In [ ]:
# Converter tipos
df['Rodada'] = df['Rodada'].astype(int)
df['Tentativa'] = df['Tentativa'].astype(int)
df['Pontuação'] = df['Pontuação'].astype(int)
df['Número de movimentos'] = df['Número de movimentos'].astype(int)

df.dtypes

## 1.1 Tradução para o inglês

Logo após a reestruturação, os **nomes das colunas** e os **valores categóricos** (`sex`, `education`) são traduzidos para o inglês. As siglas de estado (UF), as datas e a descrição de movimentos (códigos) são mantidas como estão. A partir daqui, todo o notebook usa o esquema em inglês.

In [ ]:
# Mapa de tradução dos nomes das colunas (PT -> EN)
column_translation = {
    'ID': 'id',
    'Sexo': 'sex',
    'Data de nascimento': 'birth_date',
    'Idade': 'age',
    'Escolaridade': 'education',
    'Estado onde está ocorrendo a aplicação': 'state',
    'Data de aplicação': 'application_date',
    'Rodada': 'round',
    'Tentativa': 'attempt',
    'Pontuação': 'score',
    'Tempo de início': 'start_time',
    'Tempo de fim': 'end_time',
    'Número de movimentos': 'n_moves',
    'Descrição dos movimentos': 'moves_description',
    'Tempo do primeiro clique correto': 'first_correct_click_time',
    'Tempo do primeiro clique errado': 'first_wrong_click_time',
}

df = df.rename(columns=column_translation)

# Tradução dos valores categóricos
sex_translation = {
    'Feminino': 'Female',
    'Masculino': 'Male',
}

education_translation = {
    'Não Alfabetizado': 'Illiterate',
    'Ensino Fundamental Incompleto': 'Incomplete Elementary School',
    'Ensino Fundamental Completo': 'Complete Elementary School',
    'Ensino Médio Incompleto': 'Incomplete High School',
    'Ensino Médio Completo': 'Complete High School',
    'Técnico/Profissionalizante Incompleto': 'Incomplete Technical/Vocational',
    'Técnico/Profissionalizante Completo': 'Complete Technical/Vocational',
    'Ensino Superior Incompleto': 'Incomplete Higher Education',
    'Ensino Superior Completo': 'Complete Higher Education',
    'Pós-Graduação: Especialização': 'Graduate: Specialization',
    'Pós-Graduação: Mestrado': 'Graduate: Master',
    'Pós-Graduação: Doutorado': 'Graduate: Doctorate',
}

df['sex'] = df['sex'].map(sex_translation).fillna(df['sex'])
df['education'] = df['education'].map(education_translation).fillna(df['education'])

print('Columns:', list(df.columns))
df.head()

In [ ]:
# Salvar dataset reestruturado (em inglês)
# df.to_csv('../data/interim/tol_long.csv', index=False, encoding='utf-8-sig')
# print('Salvo: ../data/interim/tol_long.csv')

## 2 Diagnóstico de Qualidade

O diagnóstico é realizado sobre a base já reestruturada no formato long e traduzida para o inglês, em que cada linha representa uma tentativa de um participante.

### 2.1 Dimensões do dataset

Verificação da quantidade de registros (tentativas), variáveis e participantes distintos presentes na base.

In [ ]:
print(f'Linhas (tentativas): {df.shape[0]:,}')
print(f'Colunas: {df.shape[1]:,}')
print(f'Participantes distintos: {df["id"].nunique():,}')
print('Shape:', df.shape)

### 2.2 Tipos de dados

Análise dos tipos das variáveis. A verificação é necessária porque valores numéricos podem estar armazenados como texto (por exemplo, os tempos com vírgula decimal) e datas podem estar representadas como strings.

In [ ]:
df.info()

### 2.3 Estatísticas descritivas

Visão inicial da distribuição das variáveis numéricas.

In [ ]:
df.describe()

### 2.4 Valores ausentes

A presença de valores ausentes é analisada em quantidade absoluta e percentual.

In [ ]:
ausentes = pd.DataFrame({
    'ausentes': df.isnull().sum(),
    'percentual (%)': (df.isnull().mean() * 100).round(2)
})

ausentes.sort_values('percentual (%)', ascending=False)

### 2.5 Duplicatas

Identificação de duplicatas completas, ou seja, linhas em que todas as colunas possuem exatamente os mesmos valores.

In [ ]:
print('Duplicatas completas:', df.duplicated().sum())

### 2.6 Inconsistências em variáveis categóricas

Inspeção das principais variáveis categóricas (já traduzidas) para verificar diferenças de grafia ou categorias equivalentes.

In [ ]:
colunas_categoricas = [
    'sex',
    'education',
    'state'
]

for col in colunas_categoricas:
    print(f'\n--- {col} ---')
    print(df[col].value_counts(dropna=False))

### 2.7 Verificação de espaços extras

Verificação da existência de espaços no início ou no final dos valores textuais.

In [ ]:
for col in colunas_categoricas:
    valores = df[col].dropna().astype(str)
    com_espacos = valores[valores.str.strip() != valores]
    print(f'{col}: {len(com_espacos)} valores com espaços extras')

### 2.8 Verificação dos formatos de data

As variáveis de data são inspecionadas para verificar se seguem um padrão consistente (dd/mm/aaaa).

In [ ]:
colunas_data = [
    'birth_date',
    'application_date'
]

for col in colunas_data:
    print(f'\n--- {col} ---')
    print(df[col].dropna().astype(str).head(10).tolist())

### 2.9 Identificação de outliers

Os outliers são investigados nas variáveis numéricas disponíveis, utilizando a regra do IQR (intervalo interquartil).

In [ ]:
def calcular_iqr(serie):
    serie = pd.to_numeric(serie, errors='coerce').dropna()

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    outliers = (serie < limite_inferior) | (serie > limite_superior)

    return {
        'Q1': q1,
        'Q3': q3,
        'IQR': iqr,
        'Limite inferior': limite_inferior,
        'Limite superior': limite_superior,
        'Quantidade': outliers.sum(),
        'Percentual (%)': outliers.mean() * 100
    }


variaveis_outlier = [
    'age',
    'round',
    'attempt',
    'score',
    'n_moves'
]

resultado_outliers = pd.DataFrame({
    col: calcular_iqr(df[col])
    for col in variaveis_outlier
}).T

resultado_outliers

### 2.10 Análise de assimetria

A assimetria das variáveis numéricas é analisada pelo coeficiente de skewness.

In [ ]:
skewness = df[variaveis_outlier].apply(pd.to_numeric, errors='coerce').skew()

skewness.sort_values(ascending=False)

### 2.11 Verificação de inconsistências entre variáveis

Verificação de combinações logicamente incompatíveis: idade muito baixa associada a escolaridade avançada pode indicar erro de preenchimento.

In [ ]:
escolaridade_avancada = [
    'Graduate: Specialization',
    'Graduate: Master',
    'Graduate: Doctorate',
    'Complete Higher Education',
    'Incomplete Higher Education'
]

inconsistencia_idade = (
    (pd.to_numeric(df['age'], errors='coerce') <= 10) &
    (df['education'].isin(escolaridade_avancada))
)

print(
    'Registros com idade <= 10 e escolaridade avançada:',
    inconsistencia_idade.sum()
)

### 2.12 Observação das variáveis de tempo

As variáveis de tempo (`start_time`, `end_time`, `first_correct_click_time`, `first_wrong_click_time`) originalmente usam vírgula como separador decimal. `start_time` e `end_time` são timestamps, enquanto os campos de clique representam durações. A conversão numérica e o cálculo da duração são feitos na etapa de limpeza/processamento.

### 2.13 Conclusão do diagnóstico

A base reestruturada e traduzida possui cerca de 960.687 tentativas e 16 variáveis, correspondendo a ~32.770 participantes distintos. Concentram-se ausências em `state` e nos tempos de clique (`first_correct_click_time`, `first_wrong_click_time`), que refletem a não ocorrência do evento. Não há duplicatas completas; as categóricas (`sex`, `education`) não têm espaços extras; datas em dd/mm/aaaa. Outliers relevantes apenas em `age`. Há registros com `age <= 10` e escolaridade avançada a investigar. Os tempos ainda precisam ser convertidos e a duração calculada na próxima etapa.

### 2.14 Salvar resultado

A base reestruturada e traduzida é salva em CSV para as próximas etapas.

In [ ]:
caminho_saida = '../data/interim/tol_long_diagnosticado.csv'

df.to_csv(caminho_saida, index=False, encoding='utf-8-sig')

print(f'Arquivo salvo: {caminho_saida}')
print(f'Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}')
print('Colunas:', list(df.columns))